# Redukcja wymiarowości danych

---
## 1. Wprowadzenie

Na początek warto uświadomić sobie, jak szybko „zaklęcie wymiarowości” potrafi skomplikować pracę z danymi. Wyobraźmy sobie przestrzeń wektorową o kilkudziesięciu czy kilkuset wymiarach – każda obserwacja staje się punktem w ogromnym hipersześcianie. W miarę wzrostu liczby cech obszar tej przestrzeni rośnie wykładniczo, a dostępne dane stają się w niej coraz bardziej rozproszone. W praktyce oznacza to, że typowe metryki odległości (np. euklidesowa) tracą znaczenie – sąsiedzi w tak rozrzedzonej przestrzeni bywają równie odlegli, co zupełnie losowe punkty. Dodatkowo operacje na macierzach o bardzo dużych wymiarach (czy to obliczanie macierzy kowariancji, czy odległości między wszystkimi parami punktów) stają się nie tylko trudne do zrealizowania pamięciowo, ale i czasochłonne, co gwałtownie podnosi koszty algorytmiczne nawet przy umiarkowanych wielkościach próby.

Redukcja wymiarowości to właśnie zestaw technik mających na celu uproszczenie tej przestrzeni. Przez transformację danych z przestrzeni oryginalnej do przestrzeni o niższym wymiarze dążymy do zachowania najistotniejszych wzorców – czyli takiej reprezentacji, w której ciągle odtwarzamy większość informacji zawartej w surowych cechach, ale eliminujemy redundancję i szum. Dzięki temu jesteśmy w stanie przyspieszyć działanie algorytmów uczenia maszynowego, uniknąć zjawiska przeuczenia (overfittingu) wynikającego z nadmiaru parametrów oraz łatwiej zwizualizować skomplikowane struktury danych, co ma ogromne znaczenie przy eksploracyjnej analizie klastrów czy identyfikacji anomalii.

W realnych zastosowaniach redukcja wymiarowości pojawia się w bardzo różnych kontekstach: od kompresji obrazów (np. wideo czy zdjęcia, gdzie transformacje podobne do PCA są podstawą standardów takich jak JPEG), przez wstępną obróbkę przed segmentacją tekstury w analizie obrazów medycznych, aż po usprawnienie rekomendacji w systemach e-commerce – tam, gdzie dane użytkownika mogą mieć tysiące cech opisujących zachowania czy preferencje. Podobnie w bioinformatyce, gdzie liczba genów przekracza dziesiątki tysięcy, redukcja wymiarowości pozwala w łatwy sposób wyodrębnić główne źródła biologicznej zmienności, bez zgubienia kluczowych sygnałów.


Warto przyjąć również formalne spojrzenie: mamy dane reprezentowane macierzowo $X \in \mathbb{R}^{n \times d}$, gdzie $n$ to liczba obserwacji, a $d$ – pełny wymiar cech. Celem redukcji jest znalezienie odwzorowania $f: \mathbb{R}^d \to \mathbb{R}^k$ (zwykle $k \ll d$), które dla każdego wiersza $x_i$ zwraca krótszy wektor $z_i = f(x_i)$, ale tak, żeby w nowej przestrzeni „nie zginęły” istotne relacje ani struktury obecne w oryginale. Dalej przyjrzymy się, w jaki sposób odmiennie definiowane miary jakości  prowadzą do różnych algorytmów – od klasycznego PCA, przez nieliniowe embeddingi, aż po zaawansowane autoenkodery.


### 1.1 Motywacja

Redukcja wymiarowości nie jest jedynie „miłym dodatkiem” do analizy danych – to fundament dobrego projektowania każdej procedury uczenia maszynowego czy eksploracji wielowymiarowych zbiorów. Poniżej znajdują się dwa kluczowe zagadnienia, które uzasadniają potrzebę redukcji wymiarowości i pokazują, gdzie jej efekt staje się nieoceniony.

#### 1. Klątwa wymiarowości

Wzrost liczby cech (wymiarów) powoduje, że przestrzeń, w której pracujemy, rozrasta się wykładniczo. Już przy kilkudziesięciu wymiarach nawet milion punktów staje się „rozrzucony” po ogromnej objętości, co rodzi trzy główne problemy:

- **Rozrzedzenie danych i koncentracja odległości**
  Przy rosnącym $d$ stosunek odległości między najbliższym a najdalszym sąsiadem zmierza ku 1 – oznacza to, że wszystkie punkty stają się równie odległe. W praktyce:
  - Metryki odległości (Euklidesowa, Mahalanobisa) przestają rozróżniać punkty.
  - Algorytmy bazujące na sąsiadach (k-NN, drzewa ball-tree, k-d tree) tracą precyzję, a wyszukiwanie najbliższych sąsiadów staje się niestabilne.

- **Eksplozja złożoności obliczeniowej**
  - Obliczenie macierzy kowariancji wymaga $\mathcal{O}(n d^2)$ operacji, gdzie $n$ to liczba obserwacji, $d$ – wymiar.
  - Przechowywanie pełnej macierzy odległości pomiędzy wszystkimi parami punktów kosztuje $\mathcal{O}(n^2)$ pamięci, a każde dodanie wymiaru wydłuża operacje wektorowe proporcjonalnie do $d$.
  - Modele liniowe (SVM, regresja liniowa) i sieci neuronowe muszą utrzymywać wysokowymiarowe wagi, co zwiększa potrzeby pamięci i liczbę gradientów do obliczenia.

- **Przeuczenie (overfitting) i potrzeba większej próby**
  - Dla modelu o $d$ cechach teoretyczna liczba parametrów rośnie liniowo (lub kwadratowo, gdy mamy interakcje), a potrzeby ilości próbek, by prawidłowo je oszacować, rosną co najmniej proporcjonalnie do $d$.
  - Gdy $n$ (próbek) ≪ $d$ (cech), model „uczy się” szumu – dopasowuje się do przypadkowych fluktuacji, zamiast do prawdziwych zależności.
  - Redukcja wymiaru pozwala zmniejszyć liczbę parametrów i przywrócić korzystny stosunek $n/d$, co ratunkowo obniża wariancję estymacji i poprawia generalizację.

#### 2. Przykłady zastosowań redukcji wymiarowości

Redukcja wymiarowości znajduje zastosowanie w bardzo różnych dziedzinach – poniżej wybrane scenariusze, w których od razu widać jej wartość.

- **Wizualizacja danych**
  - Przekształcenie danych z przestrzeni 50–200 wymiarów do 2D lub 3D (za pomocą PCA, t-SNE czy UMAP) pozwala na:
    - Wykrycie naturalnych klastrów i struktur (grupowanie tematów, segmentacja klientów).
    - Wykrycie anomalii (punkty odstające łatwo zidentyfikować na wykresie).
    - Szybkie zrozumienie ukrytych wzorców przez zmysł wzroku – kluczowe w data science i eksploracji danych.

- **Kompresja i kodowanie sygnału**
  - W standardzie JPEG obrazy są dzielone na bloki i transformowane DCT (analogia do PCA w dziedzinie częstotliwości), co pozwala:
    - Zredukować wymiar reprezentacji każdego bloku z 64 wartości do kilkunastu dominujących współczynników.
    - Zachować wizualnie istotne detale (wysoka częstotliwość to „szum” pomijany przy kompresji).
  - Podobnie w przetwarzaniu dźwięku (MP3) czy wizji komputerowej (SVD w analizie twarzy).

- **Przyspieszenie i poprawa jakości modeli ML**
  - **Zmniejszenie wymiaru przed treningiem**
    - Modele takie jak SVM, regresja logistyczna czy sieci neuronowe trenują się szybciej na danych o niskim wymiarze.
    - Mniejsza liczba cech oznacza mniej wag do nauki i krótsze czasy inferencji, co jest kluczowe w systemach czasu rzeczywistego (np. rozpoznawanie obrazów na urządzeniach mobilnych).
  - **Poprawa generalizacji**
    - Eliminacja cech słabo korelujących z celem (lub silnie skorelowanych między sobą) zmniejsza ryzyko „uczenia szumu” i poprawia zdolność modelu do uogólniania na nowe dane.
    - Redukcja wymiaru często działa jak regularizacja: obniża złożoność modelu bez konieczności dodatkowych kar (L1/L2).


Redukcja wymiarowości to niezbędny etap projektowania pipeline’u analitycznego – nie tylko wspiera wizualizację, ale przede wszystkim pozwala radzić sobie z fundamentalnymi ograniczeniami obliczeniowymi i statystycznymi, które pojawiają się wraz z rosnącą liczbą cech.


## 2. Podstawy teoretyczne

### 2.1 Rodzaje metod

#### A. Metody projekcyjne
– **Cel:** znaleźć funkcję odwzorowania
$f: \mathbb{R}^d \to \mathbb{R}^k,\quad k \ll d$
tak aby pewien miernik „jakości” projekcji był zoptymalizowany.

1. **Metody liniowe**
   - **<u>PCA</u>**
     - **Funkcja celu:** maksymalizacja wariancji składowych
       $\max_{W \in \mathbb{R}^{d\times k},\,W^TW=I}\;\mathrm{tr}\bigl(W^T \Sigma_X W\bigr)$,
       gdzie $\Sigma_X$ to macierz kowariancji danych.
     - **Wynik:** ortonormalne wektory („składowe główne”), uporządkowane według wyjaśnianej wariancji.
   - **LDA**
     - **Funkcja celu:** maksymalizacja stosunku wariancji międzyklasowej do wewnątrzklasowej
       $\max_{W}\;\frac{\det\bigl(W^T S_B W\bigr)}{\det\bigl(W^T S_W W\bigr)}$,
       gdzie $S_B$ i $S_W$ to odpowiednio macierze kowariancji między- i wewnątrzklasowe.
     - **Wynik:** przestrzeń wymiaru $\min(C-1, d)$, gdzie $C$ = liczba klas.

2. **Metody nieliniowe**
   - **t-SNE**
     - **Zasada:** minimalizacja rozbieżności między rozkładem sąsiedztwa w oryginalnej przestrzeni a rozkładem w przestrzeni niskowymiarowej.
     - **Funkcja celu (KL-divergence):**
       $\min_Y\;\sum_{i\neq j} p_{ij} \log\frac{p_{ij}}{q_{ij}}$,
       gdzie $p_{ij}$ – prawdopodobieństwo, że $i$ i $j$ są sąsiadami w oryginale, a $q_{ij}$ – w reprezentacji 2D.
     - **Parametry kluczowe:** `perplexity` (określa liczbę „rzeczywistych sąsiadów”), `learning_rate`.
   - **UMAP**
     - **Zasada:** buduje grafy „rozmyte” (fuzzy simplicial sets) w obu przestrzeniach i minimalizuje różnicę topologii.
     - **Miernik:** cross-entropy między strukturą globalno-lokalną w oryginale a w embedzie.
     - **Zalety:** zachowuje zarówno lokalną, jak i pewne cechy globalne, szybka obliczeniowo dzięki optymalizacji SGD.

3. **Metody oparte na grafach i rozmaitościach**
   - **Isomap:** zachowuje odległości geodezyjne na rozmaitości – najpierw buduje graf sąsiedztwa, potem MDS.
   - **LLE (Locally Linear Embedding):** każdą obserwację aproksymuje liniowo przez sąsiadów, potem odwzorowuje te same współczynniki w niższej przestrzeni.

---

#### B. Metody selekcyjne (wybór cech)
– **Cel:** usunąć nieistotne lub nadmiarowe cechy, zostawić podzbiór $\{X_{j_1}, \dots, X_{j_k}\}\subset\{X_1,\dots,X_d\}$.

1. **Filtracyjne**
   - **VarianceThreshold:** usuwa cechy o wariancji poniżej progu $\theta$.
   - **Korelacja/Pearson czy MI:** rankuje zmienne wg wartości $|r|$ lub miary informacji wzajemnej z celem.

2. **Oparte na modelu (Embedded)**
   - **Regularyzacja L1 (Lasso):** współczynniki częściowo wymuszone do zera.
   - **Drzewa decyzyjne / RandomForest:** cechy o niskiej „importance” odrzucane.
   - **SelectFromModel:** automatyczne wybieranie cech na podstawie ważności z wytrenowanego modelu.

3. **Wrappery (opakowujące)**
   - **Selekcja w przód (Forward Selection):** zaczyna od pustego zestawu, dodaje kolejne cechy, jeśli poprawiają walidowaną metrykę.
   - **Selekcja wstecz (Backward Elimination):** zaczyna od pełnego zestawu, usuwa cechy, których brak nie pogarsza wyniku.
   - **Metody heurystyczne:** algorytmy genetyczne, symulowane wyżarzanie – eksploracja podzbiorów.

---

### 2.2 Kryteria wyboru metody

1. **Charakter danych**
   - **Liniowe zależności** → PCA, LDA – szybkie, dobrze interpretowalne.
   - **Nieliniowe wzorce** → t-SNE, UMAP, Isomap – lepiej zachowują strukturę rozmaitości.
   - **Korekty na szum** → autoenkodery z regularyzacją (dropout, sparsity).

2. **Wielkość próby vs. liczba cech**
   - **n ≪ d** (mało próbek, dużo cech) → ryzyko overfittingu; stosować silną redukcję (PCA z n_components≪n), metody filtracyjne lub L1.
   - **n ≈ d** → LDA może być niestabilna (macierz $S_W$ osobliwa); lepiej PCA lub nieliniowe metody z regularyzacją.
   - **n ≫ d** (dużo próbek) → można zastosować kosztowne obliczeniowo metody nieliniowe (UMAP, autoenkodery).

3. **Interpretowalność wyników**
   - **Wysoka:** PCA (składowe główne jako kombinacje liniowe), LDA (wnioski o wagach cech).
   - **Średnia:** Isomap, LLE – interpretowalne przez lokalne relacje, ale brak jednoznacznych współczynników.
   - **Niska:** t-SNE, UMAP, autoenkodery – embeddingi trudne do przełożenia na oryginalne cechy bez dodatkowej analizy.

4. **Złożoność obliczeniowa**
   - **O(n d²)** i wyżej: pełne obliczenie kowariancji (PCA, LDA), Inne:
   - **O(n²)**: t-SNE (ze względu na dwumianowe prawdopodobieństwa),
   - **O(n log n)**: UMAP (skaluje się lepiej przy dużych n dzięki optymalizacji SGD i budowie grafu).

5. **Odporność na brakujące dane i outliery**
   - **Wrażliwe:** PCA, LDA – wymagają impute i eliminacji odchyleń.
   - **Mniej wrażliwe:** metody rangowe (SelectKBest mutual_info), drzewa w SelectFromModel.
   - **Autoenkodery:** można wprowadzić warstwę denoising (Denoising Autoencoder) do obsługi szumu.



## 3. PCA

### 3.1 PCA (Principal Component Analysis)

**Cel:** znaleźć nowe, ortonormalne osie (składowe główne) tak, aby pierwsza składowa wyjaśniała jak najwięcej wariancji w danych, druga wyjaśniała jak najwięcej pozostałej wariancji, itd.

1. **Krok 1 – centrowanie danych**
   $X_{\text{centered}} = X - \bar X,
     \quad
     \bar X = \frac{1}{n}\sum_{i=1}^n x_i.
   $
2. **Krok 2 – macierz kowariancji**
   $
     \Sigma = \frac{1}{n-1} X_{\text{centered}}^T X_{\text{centered}}.
   $
3. **Krok 3 – dekompozycja własna**
   $
     \Sigma = V \Lambda V^T,
   $
   gdzie kolumny $V = [v_1, \dots, v_d]$ to wektory własne, a $\Lambda$ – diagonalna macierz wartości własnych $\lambda_1 \ge \lambda_2 \ge \dots$.
4. **Transformacja**
   $
     Z = X_{\text{centered}} \, W,
     \quad
     W = [v_1, \dots, v_k],
   $
   daje nowy zbiór \(Z \in \mathbb{R}^{n\times k}\).

5. **Wskaźniki**
   - **Explained Variance Ratio**:
     $
       \text{EVR}_j = \frac{\lambda_j}{\sum_{i=1}^d \lambda_i},
    $
     mówi, jaką część całkowitej wariancji wyjaśnia \(j\)-ta składowa.

In [7]:

import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Generujemy przykładowe dane: 150 próbek, 5 cech
np.random.seed(0)
X = np.dot(np.random.randn(150, 3), np.random.randn(3, 5)) + np.random.randn(150, 5)

# 1) Standaryzacja (zalecane przed PCA)
scaler = StandardScaler()
X_std = scaler.fit_transform(X)
print(X_std.shape)

(150, 5)


In [8]:
# 2) Inicjalizacja i dopasowanie PCA
pca = PCA(n_components=2)        # redukcja do 2 wymiarów
Z = pca.fit_transform(X_std)     # Z ∈ R^{150×2}
print(Z.shape)

(150, 2)


In [9]:
# 3) Sprawdzenie wyjaśnionej wariancji
print("Explained variance ratio:", pca.explained_variance_ratio_)
# Wynik: np. [0.43395417 0.22829518] -> dwie składowe wyjaśniają ~65% wariancji
# pierwsza składowa tłumaczy 43% całkowitej wariancji oryginalnych 5 cech, druga składowa tłumaczy dodatkowe 22%.

Explained variance ratio: [0.43395417 0.22829518]


In [10]:
# 4) Wektory własne (kierunki głównych składowych)
print("Principal axes:\n", pca.components_)


Principal axes:
 [[-0.56601039  0.48936356  0.60633228 -0.17960619  0.20064476]
 [ 0.36384159  0.3431177  -0.24617989 -0.12608838  0.82060144]]


Wynikowa macierz components_ ma $k$ wierszy (składowych) i $d$ kolumn (oryginalnych cech). Każda komórka
$components_{j,i} = w_{ji}$
mówi, jak silnie cecha $i$ „wpływa” na nową składową $j$.
* Moduł wartości $|w_{ji}|$ - im większy, tym większy udział tej cechy w wariancji wyjaśnianej przez składową j.

* Znak (+/–) – określa kierunek: jeśli „ładunek” jest dodatni, wzrost oryginalnej cechy przesuwa punkt w kierunku rosnącym osi głównej składowej; jeśli ujemny – w kierunku malejącym.

Mówiąc w skrócie: cecha o dużej wadze w pierwszej składowej jest kluczowa dla „głównego” kierunku zmienności danych, a cechy z bardzo małymi wagami we wszystkich zachowywanych składowych wnoszą niewiele informacji i często można je odrzucić.

W naszym przykładzie:

Wektor [-0.56601039  0.48936356  0.60633228 -0.17960619  0.20064476]] to pierwsza składowa główna: mówi, jak „ważyć” oryginalne cechy, aby uzyskać nową o największej wariancji.

Wielkość współczynnika (moduł wartości) informuje o znaczeniu danej cechy w tej składowej:

Wartość 0.60 przy trzeciej cesze → cecha 3 silnie wpływa na pierwszą składową.

**Jak wybrać cechy do odrzucenia?**

Po wykonaniu PCA i otrzymaniu macierzy `components_` (rozmiar: `n_components × n_features`), możemy ocenić, które cechy wnoszą najmniej informacji i bezpiecznie je usunąć:

1. **Oblicz łączną „ważność” każdej cechy**
   - Weź bezwzględne wartości wag (`|w_{ji}|`) dla pierwszych $k$ składowych, które będziesz zachowywać.
   - Zsumuj je wzdłuż składowych:
     $$\text{importance}_i \;=\;\sum_{j=1}^{k}\bigl|\,\text{components\_}[j,i]\bigr|.
     $$
   - Im mniejsza wartość $\text{importance}_i$, tym mniejszy wkład cechy $i$ w wyjaśnioną wariancję.

2. **Ustal próg odrzucenia lub liczbę cech do usunięcia**
   - Wariant 1: określ procent cech o najniższej łącznej wadzę (np. 10–20%) i odrzuć je wszystkie.
   - Wariant 2: zachowaj tylko te cechy, których łączna waga przekracza ustalony próg minimalny $\tau$.

3. **Zweryfikuj wybór cech pod kątem domenowym**
   - Nawet jeśli cecha ma niską wagę w PCA, może mieć kluczowe znaczenie biznesowe lub eksperymentalne.
   - Sprawdź, czy usunięcie danej cechy nie pozbawi modelu ważnego sygnału.

4. **Przykładowe podejście**


In [19]:
W = pca.components_ # k × d
print(W)
feature_importance = np.sum(np.abs(W), axis=0)

# Posortuj cechy po ważności
indices = np.argsort(feature_importance)   # od najmniej ważnej do najważniejszej
print("Ranking cech od najmniej do najbardziej istotnych:")
for idx in indices:
    print(f"Cechą {idx} – łączna waga {feature_importance[idx]:.3f}") #feature_importance[i] to suma absolutnych wag cechy i w zachowanych składowych.
    #Małe wartości sugerują, że usunięcie tej cechy nie wywoła dużej utraty wyjaśnianej wariancji.

## Załóżmy, że odrzucamy 20% cech o najniższej ważności
n_drop = int(0.2 * X.shape[1])
to_drop = indices[:n_drop]
print("Do odrzucenia (indeksy):", to_drop)

[[-0.56601039  0.48936356  0.60633228 -0.17960619  0.20064476]
 [ 0.36384159  0.3431177  -0.24617989 -0.12608838  0.82060144]]
Ranking cech od najmniej do najbardziej istotnych:
Cechą 3 – łączna waga 0.306
Cechą 1 – łączna waga 0.832
Cechą 2 – łączna waga 0.853
Cechą 0 – łączna waga 0.930
Cechą 4 – łączna waga 1.021
Do odrzucenia (indeksy): [3]


**Kiedy użyć PCA?**

Gdy mamy dużo cech i chcemy znaleźć najważniejsze kierunki zmienności.

Do wstępnej kompresji danych przed wizualizacją lub przyspieszeniem modelu.

Gdy zakładamy, że dane „leżą blisko” podprzestrzeni liniowej

**Podsumowanie PCA**

* Ważenie cech w PCA to patrzenie na wartości z macierzy components_: im większa wartość bezwzględna, tym większy udział cechy w danej składowej.
* Cechy o bardzo małych wagach we wszystkich składowych, które zachowujesz, wnoszą relatywnie mało informacji i często można je odrzucić.
* Zawsze jednak weryfikuj biznesowe/eksperymentalne znaczenie cech — PCA pomaga wskazać kandydatów, ale nie zastępuje eksperckiej oceny.

## 4. SelectFromModel z drzewem decyzyjnym

`SelectFromModel` to meta-transformer z pakietu `scikit-learn`, który wykorzystuje wytrenowany model do oceny „ważności” każdej cechy i automatycznie odrzuca te, które uzna za mało istotne. Modelami bazowymi mogą być:

- **Lasso** (regresja z karą L1) — współczynniki niektórych cech wymuszone do zera wskazują na ich brak wpływu.
- **Drzewa decyzyjne** (DecisionTree, RandomForest) — każda cecha ma przypisaną wartość `feature_importances_` wynikającą z jej wkładu w podział danych.


`SelectFromModel` bazuje na założeniu, że dobrze dopasowany model ML sam podpowie, które cechy są istotne. W przypadku drzewiastych modeli (DecisionTree, RandomForest, ExtraTrees) wykorzystujemy atrybut `feature_importances_`, który mierzy wkład każdej cechy w redukcję nieczystości (impurity) podczas budowy drzewa.

---

### 4.1 Skąd bierze się `feature_importances_`?

1. **Impurity-based importance**
   - **Dla klasyfikacji:** miara to suma spadku entropii (lub Gini) przy każdym rozbiciu drzewa, w którym użyto danej cechy, ważona przez liczbę próbek przepływających przez to rozbicie.
   - **Dla regresji:** analogicznie, ale z użyciem spadku wariancji.
   - **Formuła (upraszczając):**
     $$\text{importance}_j = \sum_{t \in \text{nodes using feature }j} \frac{N_t}{N} \,\Delta i_t
     $$
     gdzie $N_t$ to liczba próbek w węźle $t$, $N$ – łącznie próbek, a $\Delta i_t$ – spadek miary nieczystości w tym węźle.

2. **Permutacyjne ważności (Permutation Importance)**
   - Alternatywa odporna na bias drzew (drzewa faworyzują cechy o wielu unikalnych wartościach).
   - Dla każdej cechy losowo mieszamy jej kolumnę w zbiorze walidacyjnym i mierzymy spadek metryki (np. accuracy, R²).
   - Im większy spadek, tym ważniejsza cecha.

---

### 4.2 Wybór progu (`threshold`)

- **Domyślny próg:** średnia wartość wszystkich `feature_importances_` (tylko w nowszych wersjach sklearn).
- **Opcje niestandardowe:**
  - `threshold='median'` – usuń połowę cech o najniższych importancjach.
  - `threshold=0.01` – zachowaj tylko cechy o importance ≥ 0.01.
  - `threshold='mean'` – zachowaj cechy powyżej średniej.
  - Dowolna funkcja zwracająca liczbę graniczną:


In [ ]:
# 1. Importy i przygotowanie danych
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [26]:
# Wczytanie przykładowego zbioru Iris
iris = load_iris()
feature_names = iris.feature_names  # np. ["sepal length", "sepal width", ...]
feature_names

['sepal length (cm)',
 'sepal width (cm)',
 'petal length (cm)',
 'petal width (cm)']

In [ ]:
iris.target_names

In [30]:
 iris

{'data': array([[5.1, 3.5, 1.4, 0.2],
        [4.9, 3. , 1.4, 0.2],
        [4.7, 3.2, 1.3, 0.2],
        [4.6, 3.1, 1.5, 0.2],
        [5. , 3.6, 1.4, 0.2],
        [5.4, 3.9, 1.7, 0.4],
        [4.6, 3.4, 1.4, 0.3],
        [5. , 3.4, 1.5, 0.2],
        [4.4, 2.9, 1.4, 0.2],
        [4.9, 3.1, 1.5, 0.1],
        [5.4, 3.7, 1.5, 0.2],
        [4.8, 3.4, 1.6, 0.2],
        [4.8, 3. , 1.4, 0.1],
        [4.3, 3. , 1.1, 0.1],
        [5.8, 4. , 1.2, 0.2],
        [5.7, 4.4, 1.5, 0.4],
        [5.4, 3.9, 1.3, 0.4],
        [5.1, 3.5, 1.4, 0.3],
        [5.7, 3.8, 1.7, 0.3],
        [5.1, 3.8, 1.5, 0.3],
        [5.4, 3.4, 1.7, 0.2],
        [5.1, 3.7, 1.5, 0.4],
        [4.6, 3.6, 1. , 0.2],
        [5.1, 3.3, 1.7, 0.5],
        [4.8, 3.4, 1.9, 0.2],
        [5. , 3. , 1.6, 0.2],
        [5. , 3.4, 1.6, 0.4],
        [5.2, 3.5, 1.5, 0.2],
        [5.2, 3.4, 1.4, 0.2],
        [4.7, 3.2, 1.6, 0.2],
        [4.8, 3.1, 1.6, 0.2],
        [5.4, 3.4, 1.5, 0.4],
        [5.2, 4.1, 1.5, 0.1],
  

In [25]:
X, y = load_iris(return_X_y=True)

In [22]:
X

array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2],
       [4.6, 3.1, 1.5, 0.2],
       [5. , 3.6, 1.4, 0.2],
       [5.4, 3.9, 1.7, 0.4],
       [4.6, 3.4, 1.4, 0.3],
       [5. , 3.4, 1.5, 0.2],
       [4.4, 2.9, 1.4, 0.2],
       [4.9, 3.1, 1.5, 0.1],
       [5.4, 3.7, 1.5, 0.2],
       [4.8, 3.4, 1.6, 0.2],
       [4.8, 3. , 1.4, 0.1],
       [4.3, 3. , 1.1, 0.1],
       [5.8, 4. , 1.2, 0.2],
       [5.7, 4.4, 1.5, 0.4],
       [5.4, 3.9, 1.3, 0.4],
       [5.1, 3.5, 1.4, 0.3],
       [5.7, 3.8, 1.7, 0.3],
       [5.1, 3.8, 1.5, 0.3],
       [5.4, 3.4, 1.7, 0.2],
       [5.1, 3.7, 1.5, 0.4],
       [4.6, 3.6, 1. , 0.2],
       [5.1, 3.3, 1.7, 0.5],
       [4.8, 3.4, 1.9, 0.2],
       [5. , 3. , 1.6, 0.2],
       [5. , 3.4, 1.6, 0.4],
       [5.2, 3.5, 1.5, 0.2],
       [5.2, 3.4, 1.4, 0.2],
       [4.7, 3.2, 1.6, 0.2],
       [4.8, 3.1, 1.6, 0.2],
       [5.4, 3.4, 1.5, 0.4],
       [5.2, 4.1, 1.5, 0.1],
       [5.5, 4.2, 1.4, 0.2],
       [4.9, 3

In [23]:
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [32]:
# Podział na zbiór treningowy (70%) i walidacyjny (30%)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 2. Budowa pipeline
pipeline = Pipeline([
    # Standaryzacja: cechy o różnych skalach zostaną wyrównane
    ('scaler', StandardScaler()),

    # Selekcja cech:
    # - RandomForestClassifier oblicza feature_importances_
    # - SelectFromModel odrzuca cechy poniżej mediany importance
    ('feature_sel', SelectFromModel(
        RandomForestClassifier(n_estimators=50, random_state=42),
        threshold='median'
    )),

    # Klasyfikator końcowy: regresja logistyczna
    ('clf', LogisticRegression(max_iter=1000))
])

# 3. Trenowanie całego pipeline
pipeline.fit(X_train, y_train)

# 4. Sprawdzenie liczby cech po selekcji
n_selected = pipeline.named_steps['feature_sel'] \
                    .transform(X_train).shape[1]
print("Liczba cech po selekcji:", n_selected)

# 5. Ocena dokładności na zbiorze walidacyjnym
accuracy = pipeline.score(X_val, y_val)
print("Accuracy na zbiorze walidacyjnym:", accuracy)


Liczba cech po selekcji: 2
Accuracy na zbiorze walidacyjnym: 1.0


### Jak sprawdzić, które cechy zostały wybrane?

In [33]:
# Załóżmy, że masz listę oryginalnych nazw cech:
feature_names = iris.feature_names  # np. ["sepal length", "sepal width", ...]

# 1. Pobierz krok selekcji z pipeline
selector = pipeline.named_steps['feature_sel']

# 2. Uzyskaj maskę boolean
mask = selector.get_support()
# mask to tablica np. [True, False, True, True] – True = cecha zachowana

# 3. Wypisz nazwy wybranych cech
selected_features = [name for name, keep in zip(feature_names, mask) if keep]
print("Wybrane cechy:", selected_features)

# 4. (Opcjonalnie) Indeksy wybranych cech
import numpy as np
selected_indices = np.where(mask)[0]
print("Indeksy wybranych cech:", selected_indices)


Wybrane cechy: ['petal length (cm)', 'petal width (cm)']
Indeksy wybranych cech: [2 3]




---

### 4.3 Zalety i pułapki

- **Zalety:**
  - Umożliwia automatyczną selekcję, bez ręcznego rankingu cech.
  - Model bazowy uwzględnia interakcje i nieliniowości (w przypadku lasów losowych).
  - Łatwo włączyć do `Pipeline` i łączyć z innymi przekształceniami.

- **Pułapki:**
  - **Bias wysokowartościowy:** cechy numeryczne lub o dużej wariancji często mają wyższą importance.
  - **Korelacja cech:** jeśli dwie cechy silnie skorelowane, drzewo może wsadzić całą wagę w jedną z nich, a druga zostanie odrzucona („maskowanie” cech).
  - **Niestabilność:** pojedyncze drzewo może dawać różne importances, dlatego lepiej użyć `RandomForest` lub `ExtraTrees` z wieloma drzewami.

